# Blocking experiments

Use this notebook to compare candidate generation settings on a reproducible sample of training Source 1 records. Reusable normalization, data loading, retrieval, and evaluation code lives under `src/`. The selected approach can later be exposed through the model pipeline and a saved config.

Runs search the full Source 2 and Source 3 corpus and can be expensive. Start with a small `s1_sample_divisor` sample and modest `k_values`. No external business lookup is used.

In [1]:
from copy import deepcopy
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipelines.blocking_experiment import run_blocking_experiment

base_config = json.loads((ROOT / 'configs/blocking_char_tfidf.json').read_text())
base_config

{'pipeline': 'blocking',
 'experiment_name': 'char_tfidf_v1',
 'dataset': 'data/raw/student_resource/dataset',
 'blocking': {'method': 'char_tfidf',
  'field': 'business_name',
  'ngram_range': [2, 5],
  'min_df': 2,
  'n_features': 262144,
  'k_values': [100, 200, 1000, 2000, 5000],
  's1_sample_divisor': 4000,
  'chunk_size': 50000,
  'query_batch_size': 16,
  'normalization': {'casefold': True,
   'unicode_nfkc': True,
   'punctuation_to_space': True,
   'collapse_whitespace': True}}}

## Define configurations to compare

Keep a single setting change per variant where practical. These examples compare retrieval depth and name n-grams; adjust them based on the missed-link and candidate-volume diagnostics.

In [2]:
variants = {
    'current': {},
    'larger_k': {'blocking': {'k_values': [200, 500, 1000], 's1_sample_divisor': 4000}},
    'name_ngrams_3_6': {'blocking': {'ngram_range': [3, 6], 'k_values': [200, 500, 1000], 's1_sample_divisor': 4000}},
}

def with_overrides(config, overrides):
    result = deepcopy(config)
    for section, values in overrides.items():
        result.setdefault(section, {}).update(values)
    return result

## Run selected variants

Uncomment or add variants deliberately; each full-corpus run may take substantial time. Outputs are saved in timestamped directories under `outputs/experiments/<experiment_name>/`.

In [ ]:
selected = ['current']  # Add variants after deciding to run them.
results = {}
for name in selected:
    config = with_overrides(base_config, variants[name])
    config['experiment_name'] = f"{base_config['experiment_name']}_{name}"
    results[name] = run_blocking_experiment(config, ROOT)

results

## Compare and inspect misses

Candidate recall and full-hit rate describe the blocking ceiling; average candidate count indicates how much work the matcher would receive. Inspect the missed-pair reports before selecting a blocker.

In [ ]:
import pandas as pd

comparison = pd.concat(
    [pd.DataFrame(value['comparison']).assign(variant=name) for name, value in results.items()],
    ignore_index=True,
) if results else pd.DataFrame()
comparison

In [ ]:
# Example: inspect missed true links from one completed run.
# run_dir = Path(results['current']['candidate_artifact']).parent
# missed = pd.read_csv(run_dir / 'missed_pairs.tsv', sep='\t')
# missed.head()